## Compare 3 versions of SCB Myndighetsregistret
* [#59](https://github.com/salgo60/SCB-Wikidata/issues/59)
* [Dumped versions](https://github.com/salgo60/SCB-Wikidata/tree/main/notebook/Myndighetsreg_concat)
   * [Nov 21, 2025 raw](https://raw.githubusercontent.com/salgo60/SCB-Wikidata/800f6c3c4430abd24027b8178e2b7cf5cf545088/notebook/Myndighetsreg_concat/Myndighetsreg_alla.csv)
   * [Jan 12, 2026 raw](https://raw.githubusercontent.com/salgo60/SCB-Wikidata/refs/heads/main/notebook/Myndighetsreg_concat/Myndighetsreg_alla_20260112.csv)
   * [Jan 26_2026 raw](https://raw.githubusercontent.com/salgo60/SCB-Wikidata/main/notebook/Myndighetsreg_concat/Myndighetsreg_alla_20260126.csv)
   * [Maj 3 2026 raw](https://raw.githubusercontent.com/salgo60/SCB-Wikidata/main/notebook/Myndighetsreg_concat/Myndighetsreg_alla_20260503.csv)
* Notebook [Myndighetsregistret_versions_59_3](https://github.com/salgo60/SCB-Wikidata/tree/800f6c3c4430abd24027b8178e2b7cf5cf545088/notebook/Myndighetsregistret_versions_59_3.ipynb)

We skip all organisations without "Organisationsnr" the register has bad quality and no unique identifier

In [1]:
import time

from datetime import datetime

now = datetime.now()
timestamp = now.timestamp()

start_time = time.time()
print("Start:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Start: 2026-05-03 12:08:21


In [2]:
import os

# Get the current working directory
current_directory = os.getcwd()
print("Current Working Directory:", current_directory)



Current Working Directory: /Users/salgo/Documents/GitHub/SCB-Wikidata/notebook


In [5]:
# --- 1. Hitta alla filer i katalogen ---
DIR = 'myndighetsreg_export/20260503'
files = [os.path.join(DIR, f) for f in os.listdir(DIR) if f.lower().endswith('.txt')]
#today = date.today().strftime("%Y_%m_%d")

print("Hittade filer:")
for f in files:
    print(f)

Hittade filer:
myndighetsreg_export/20260503/Statliga förvaltningsmyndigheter.txt
myndighetsreg_export/20260503/Svenska utlandsmyndigheter.txt
myndighetsreg_export/20260503/AP-fonder.txt
myndighetsreg_export/20260503/Statliga affärsverk.txt
myndighetsreg_export/20260503/Sveriges domstolar samt Domstolsverket.txt
myndighetsreg_export/20260503/Myndigheter under riksdagen.txt


### Skapa en fil för hela databasen

In [6]:
dateGenerated = "20260503"

In [7]:
import os
import pandas as pd

DIR = "Myndighetsreg_export/" + dateGenerated

# Hitta alla txt-filer
files = [os.path.join(DIR, f) for f in os.listdir(DIR) if f.lower().endswith('.txt')]

dfs = []

for file in files:
    # Justera sep beroende på filtyp: '\t', ';', ',' osv.
    df = pd.read_csv(file, sep="\t", encoding="utf-8")  
    dfs.append(df)

# Slå ihop till en enda DataFrame
merged = pd.concat(dfs, ignore_index=True)

# Spara som en CSV
merged.to_csv("Myndighetsreg_concat/Myndighetsreg_alla_" + dateGenerated + ".csv", index=False, encoding="utf-8")

print("Klart! skapade sammanstallt " + dateGenerated + ".csv")

Klart! skapade sammanstallt 20260503.csv


### Ladda upp skapad fil till GITHUB

In [9]:
import pandas as pd
from difflib import SequenceMatcher
from IPython.display import display

v1_url = "https://raw.githubusercontent.com/salgo60/SCB-Wikidata/main/notebook/Myndighetsreg_concat/Myndighetsreg_alla.csv"
v2_url = "https://raw.githubusercontent.com/salgo60/SCB-Wikidata/main/notebook/Myndighetsreg_concat/Myndighetsreg_alla_" + dateGenerated + ".csv"
print(v2_url)
df1 = pd.read_csv(v1_url, dtype=str).fillna("")
df2 = pd.read_csv(v2_url, dtype=str).fillna("")

key = "Organisationsnr"
name_col = "Namn"

# Drop junk rows without OrgNr
df1 = df1[df1[key].str.strip() != ""]
df2 = df2[df2[key].str.strip() != ""]

print("After OrgNr filter:")
print("v1:", len(df1), "v2:", len(df2))


https://raw.githubusercontent.com/salgo60/SCB-Wikidata/main/notebook/Myndighetsreg_concat/Myndighetsreg_alla_20260503.csv
After OrgNr filter:
v1: 347 v2: 341


In [10]:
def canonicalize(df):
    return (
        df
        .sort_values(name_col, key=lambda s: s.str.len(), ascending=False)
        .drop_duplicates(key)
        .set_index(key)
    )

df1c = canonicalize(df1)
df2c = canonicalize(df2)

print("Canonical v1:", len(df1c))
print("Canonical v2:", len(df2c))


Canonical v1: 267
Canonical v2: 261


In [11]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 347 entries, 0 to 454
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Organisationsnr           347 non-null    object
 1   Namn                      347 non-null    object
 2   PostAdress                347 non-null    object
 3   PostNr                    347 non-null    object
 4   PostOrt                   347 non-null    object
 5   BesöksAdress              347 non-null    object
 6   BesöksPostNr              347 non-null    object
 7   BesöksPostOrt             347 non-null    object
 8   Tfn                       347 non-null    object
 9   Fax                       347 non-null    object
 10  Epost                     347 non-null    object
 11  Webbadress                347 non-null    object
 12  SFS                       347 non-null    object
 13  Land                      347 non-null    object
 14  Ambassadör/Generalkonsul  347 n

In [12]:
import pandas as pd
import numpy as np
from IPython.display import display
from difflib import SequenceMatcher

v1_url = "https://raw.githubusercontent.com/salgo60/SCB-Wikidata/main/notebook/Myndighetsreg_concat/Myndighetsreg_alla.csv"
v2_url = "https://raw.githubusercontent.com/salgo60/SCB-Wikidata/main/notebook/Myndighetsreg_concat/Myndighetsreg_alla_" + dateGenerated + ".csv"

df1 = pd.read_csv(v1_url, dtype=str).fillna("")
df2 = pd.read_csv(v2_url, dtype=str).fillna("")

print(len(df1), "rows in v1")
print(len(df2), "rows in v2")

key = "Organisationsnr"
name_col = "Namn"

df1 = df1.set_index(key)
df2 = df2.set_index(key)

print("v1:", len(df1), "v2:", len(df2))

ids1 = set(df1.index)
ids2 = set(df2.index)

# Drop junk rows without OrgNr
#df1 = df1[df1[key].str.strip() != ""]
#df2 = df2[df2[key].str.strip() != ""]

#print("Lacks Organisationsnr:", len(df1), "v2:", len(df2))



deleted_ids = ids1 - ids2
added_ids   = ids2 - ids1
common_ids  = ids1 & ids2

print("Deleted:", len(deleted_ids))
print("Added:", len(added_ids))
print("Common:", len(common_ids))



455 rows in v1
449 rows in v2
v1: 455 v2: 449
Deleted: 6
Added: 0
Common: 262


In [13]:
def canonicalize(df, name_col):
    return (
        df
        .reset_index()
        .sort_values(name_col, key=lambda s: s.str.len(), ascending=False)
        .drop_duplicates("Organisationsnr")
        .set_index("Organisationsnr")
    )

df1c = canonicalize(df1.reset_index(), name_col)
df2c = canonicalize(df2.reset_index(), name_col)

print("Canonical v1:", len(df1c))
print("Canonical v2:", len(df2c))


Canonical v1: 268
Canonical v2: 262


In [14]:
ids1 = set(df1c.index)
ids2 = set(df2c.index)

deleted_ids = ids1 - ids2
added_ids   = ids2 - ids1
common_ids  = ids1 & ids2

print("Deleted:", len(deleted_ids))
print("Added:", len(added_ids))
print("Common:", len(common_ids))


Deleted: 6
Added: 0
Common: 262


In [15]:
deleted_ids, added_ids

({'202100-0464',
  '202100-0852',
  '202100-1033',
  '202100-3427',
  '202100-5141',
  '202100-6875'},
 set())

In [16]:
df_deleted_named = (
    df1c
    .loc[list(deleted_ids), [name_col, "SFS"]]
    .reset_index()
    .rename(columns={
        "Organisationsnummer": "OrgNr",
        name_col: "Namn"
    })
    .sort_values("Namn")
)

display(df_deleted_named)

,Organisationsnr,Namn,SFS
0,202100-6875,MYNDIGHETEN FÖR ARBETSMILJÖKUNSKAP,2018:254
2,202100-5141,Myndigheten för stöd till trossamfund,2017:104
5,202100-0464,STATENS FÖRSVARSHISTORISKA MUSEER,2007:1178
3,202100-1033,STATENS KONSTRÅD,2007:1188
4,202100-0852,STATSKONTORET,2025:681
1,202100-3427,Statens centrum för arkitektur och design,2013:71


In [17]:
from difflib import SequenceMatcher

rename_rows = []

for org in common_ids:
    old = df1c.loc[org, name_col]
    new = df2c.loc[org, name_col]

    if old != new:
        sim = SequenceMatcher(None, old.lower(), new.lower()).ratio()
        rename_rows.append([org, old, new, round(sim,3)])

df_renames = pd.DataFrame(
    rename_rows,
    columns=["OrgNr","Old name","New name","Similarity"]
).sort_values("Similarity")

display(df_renames)



,OrgNr,Old name,New name,Similarity
1,202100-5026,EKONOMISTYRNINGSVERKET,Statskontoret,0.400
3,202100-5984,Myndigheten för samhällsskydd och beredskap (MSB),Myndigheten för civilt försvar,0.481
0,202100-1132,STATENS MARITIMA OCH TRANSPORTHISTORISKA MUSEER,"STATENS MUSEER FÖR MARITIM, TRANSPORT- OCH FÖR...",0.629
4,202100-5703,Säkerhets- och integritetsskyddsnämnden,Myndigheten för säkerhet och integritetsskydd,0.690
2,202100-2742,FÖRVALTNINGSRÄTTEN I STOCKHOLM OCH MIGRATIONSDOM,FÖRVALTNINGSRÄTTEN I GÖTEBORG OCH MIGRATIONSDOMS,0.854


In [18]:
reorg_matches = []

for del_id in deleted_ids:
    old_name = df1c.loc[del_id, name_col]

    for add_id in added_ids:
        new_name = df2c.loc[add_id, name_col]
        sim = SequenceMatcher(None, old_name.lower(), new_name.lower()).ratio()

        if sim > 0.6:
            reorg_matches.append([del_id, old_name, add_id, new_name, round(sim,3)])

df_reorg = pd.DataFrame(
    reorg_matches,
    columns=["Old OrgNr","Old name","New OrgNr","New name","Similarity"]
).sort_values("Similarity", ascending=False)

display(df_reorg)


,Old OrgNr,Old name,New OrgNr,New name,Similarity


In [19]:
splits  = df_reorg.groupby("Old OrgNr").filter(lambda x: len(x) > 1)
mergers = df_reorg.groupby("New OrgNr").filter(lambda x: len(x) > 1)

display(splits)
display(mergers)


,Old OrgNr,Old name,New OrgNr,New name,Similarity


,Old OrgNr,Old name,New OrgNr,New name,Similarity


In [22]:
classification = []

for org in deleted_ids:
    if org in splits["Old OrgNr"].values:
        status = "Split"
    elif org in df_reorg["Old OrgNr"].values:
        status = "Merged / Renamed"
    else:
        status = "Closed"

    classification.append([org, df1c.loc[org,name_col], status])

df_deleted_classified = pd.DataFrame(
    classification,
    columns=["OrgNr","Name","Status"]
)

display(df_deleted_classified)


,OrgNr,Name,Status
0,202100-6875,MYNDIGHETEN FÖR ARBETSMILJÖKUNSKAP,Closed
1,202100-3427,Statens centrum för arkitektur och design,Closed
2,202100-5141,Myndigheten för stöd till trossamfund,Closed
3,202100-1033,STATENS KONSTRÅD,Closed
4,202100-0852,STATSKONTORET,Closed
5,202100-0464,STATENS FÖRSVARSHISTORISKA MUSEER,Closed


In [23]:
 # End timer and calculate duration
end_time = time.time()
elapsed_time = end_time - start_time# Bygg audit-lager för den här etappen

# Print current date and total time
print("Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
minutes, seconds = divmod(elapsed_time, 60)
print("Total time elapsed: {:02.0f} minutes {:05.2f} seconds".format(minutes, seconds))


Date: 2026-05-03 12:39:34
Total time elapsed: 31 minutes 13.16 seconds
